# 🧠 DocMind — Multi-Modal RAG System
**DSAI 413 — Assignment 1**

This notebook runs the full Multi-Modal RAG pipeline on Google Colab with free GPU.

**Steps:**
1. Install dependencies
2. Clone/upload your project
3. Ingest PDFs with ColPali
4. Launch Gradio chat interface
5. Run evaluation suite

## 1. Setup & Install

In [ ]:
# Install system dependencies
!apt-get update -qq && apt-get install -y -qq poppler-utils

# Install Python dependencies
!pip install -q colpali-engine torch torchvision transformers \
    pdf2image pdfplumber qdrant-client google-generativeai openai \
    streamlit gradio rouge-score sentence-transformers \
    kagglehub python-dotenv tqdm loguru Pillow

In [ ]:
# Clone your repo (replace with your GitHub URL)
# !git clone https://github.com/YOUR_USERNAME/multimodal_rag.git
# %cd multimodal_rag

# OR upload your project files manually and set the path:
import os
os.chdir('/content')  # adjust if needed

# Upload project zip and extract:
# from google.colab import files
# uploaded = files.upload()  # upload multimodal_rag.zip
# !unzip multimodal_rag.zip -d multimodal_rag
# %cd multimodal_rag

## 2. Set API Keys

In [ ]:
import os
from google.colab import userdata

# Option A: Set from Colab secrets (recommended)
# Go to the key icon in the left sidebar to add secrets
try:
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('✓ GEMINI_API_KEY loaded from Colab secrets')
except:
    pass

# Option B: Set directly (not recommended for sharing)
# os.environ['GEMINI_API_KEY'] = 'your-key-here'

os.environ['LLM_BACKEND'] = 'gemini'
print(f'Backend: {os.environ.get("LLM_BACKEND")}')

## 3. Upload & Ingest PDFs

In [ ]:
# Upload PDFs directly
from google.colab import files as colab_files
import shutil
from pathlib import Path

pdf_dir = Path('data/pdfs')
pdf_dir.mkdir(parents=True, exist_ok=True)

print('Upload your PDF files:')
uploaded = colab_files.upload()
for name, data in uploaded.items():
    dest = pdf_dir / name
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  ✓ Saved: {dest}')

print(f'\nTotal PDFs: {len(list(pdf_dir.glob("*.pdf")))}')

In [ ]:
# OR download from Kaggle
# from src.ingestion import download_kaggle_dataset
# download_kaggle_dataset()

In [ ]:
# Run ingestion pipeline
from src.ingestion import ColPaliIngester

ingester = ColPaliIngester(
    index_name='rag_index',
    dpi=150,
    store_page_images=True,
)

results = ingester.ingest_directory(Path('data/pdfs'))
ingester.save_manifest(results)

print('\n=== Ingestion Results ===')
for doc, pages in results.items():
    print(f'  {doc}: {pages} pages')

## 4. Test Retrieval & Generation

In [ ]:
from src.retrieval import Retriever
from src.generation import get_generator
from IPython.display import display

retriever = Retriever(index_name='rag_index')
generator = get_generator('gemini')

# Test query
query = 'What is the main topic of the document?'
pages = retriever.search(query, top_k=3)

print(f'Query: {query}\n')
print(f'Retrieved {len(pages)} pages:')
for p in pages:
    print(f'  {p.citation} (score: {p.score:.3f})')

# Display thumbnails
for p in pages[:3]:
    if p.page_image:
        print(f'\n--- {p.citation} ---')
        display(p.page_image.resize((400, 520)))

# Generate answer
answer = generator.generate(query, pages)
print(f'\n=== Answer ===\n{answer}')

## 5. Launch Gradio Demo

In [ ]:
from demo_gradio import create_demo

demo = create_demo()
demo.launch(share=True, debug=True)

## 6. Run Evaluation Suite

In [ ]:
from src.evaluate import Evaluator
import json

evaluator = Evaluator(
    index_name='rag_index',
    top_k=5,
    llm_backend='gemini',
)

results = evaluator.run()
summary = evaluator.summarise(results)

print('\n=== EVALUATION SUMMARY ===')
for modality, metrics in summary.items():
    print(f'\n[{modality.upper()}]')
    for k, v in metrics.items():
        print(f'  {k}: {v}')

# Save results
from dataclasses import asdict
with open('data/eval_results.json', 'w') as f:
    json.dump({'summary': summary, 'results': [asdict(r) for r in results]}, f, indent=2)
print('\nResults saved to data/eval_results.json')